# Tutorial 2: Working with Chunks and Declarative Memory

## Table of Contents
1. [Chunks: The Building Blocks of Knowledge](#chunks)
2. [Declarative Memory Mechanics](#declarative-memory)
3. [Activation and Retrieval](#activation-retrieval)
4. [Forgetting and Memory Decay](#forgetting)
5. [Real-World Application: List Learning](#list-learning)
6. [Exercises](#exercises)

## Introduction

In Tutorial 1, we used chunks as simple counting facts. But chunks can represent any kind of knowledge: semantic facts, episodic memories, conceptual relationships, even goals and intentions.

ACT-R's declarative memory isn't just a storage system. It models how human memory actually works: we forget things, confuse similar items, and retrieve information at different speeds. The system uses mathematical equations to capture activation dynamics, memory decay, and retrieval probability.

This tutorial covers how to build realistic memory models in pyactr.

In [ ]:
import pyactr as actr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from collections import defaultdict

np.random.seed(42)

## Chunks: The Building Blocks of Knowledge <a id='chunks'></a>

Chunks can represent:
- Facts (Paris is the capital of France)
- Episodes (I had coffee this morning)
- Concepts (A dog is a mammal)
- Goals (I need to buy groceries)

Here are different ways to create and work with chunks.

In [ ]:
# Define chunk types for different kinds of knowledge

actr.chunktype("fact", ("subject", "relation", "object"))
actr.chunktype("episode", ("event", "time", "location", "emotion"))
actr.chunktype("word", ("form", "meaning", "category", "frequency"))
actr.chunktype("person", ("name", "age", "occupation", "relationship"))

In [ ]:
# Create various chunks

paris_fact = actr.makechunk(typename="fact", 
                           subject="Paris", 
                           relation="capital-of", 
                           object="France")

rome_fact = actr.makechunk(typename="fact",
                          subject="Rome",
                          relation="capital-of",
                          object="Italy")

breakfast = actr.makechunk(typename="episode",
                          event="ate-breakfast",
                          time="morning",
                          location="kitchen",
                          emotion="happy")

word_dog = actr.makechunk(typename="word",
                         form="dog",
                         meaning="canine-animal",
                         category="noun",
                         frequency="high")

friend = actr.makechunk(typename="person",
                       name="Alice",
                       age=25,
                       occupation="engineer",
                       relationship="friend")

print("Chunks created!")
print(f"\nParis fact: {paris_fact}")
print(f"Breakfast episode: {breakfast}")
print(f"Word 'dog': {word_dog}")

# Access chunk attributes
print(f"\nThe subject of paris_fact is: {paris_fact.subject}")
print(f"Alice's occupation is: {friend.occupation}")

### Creating Chunks from Strings

For complex models, you can create chunks from formatted strings. This is useful when loading from external files.

In [ ]:
# chunkstring returns a list even for single chunks
london_fact = actr.chunkstring(string="""
    isa     fact
    subject London
    relation capital-of
    object  UK
""")[0]

geography_facts = [london_fact]

more_capitals = [
    ("Berlin", "Germany"),
    ("Madrid", "Spain"),
    ("Tokyo", "Japan"),
    ("Ottawa", "Canada"),
    ("Canberra", "Australia")
]

for city, country in more_capitals:
    chunk = actr.makechunk(typename="fact",
                          subject=city,
                          relation="capital-of",
                          object=country)
    geography_facts.append(chunk)

print(f"Created {len(geography_facts)} geography facts")
for fact in geography_facts:
    print(f"  {fact}")

## Declarative Memory Mechanics <a id='declarative-memory'></a>

Declarative memory in ACT-R is an active system that:
- Tracks when chunks were created and used
- Calculates activation levels
- Determines retrieval probability and speed

In [ ]:
memory_model = actr.ACTRModel()

# Enable subsymbolic processing to activate mathematical memory equations
memory_model.model_parameters["subsymbolic"] = True
memory_model.model_parameters["decay"] = 0.5  # higher = faster forgetting
memory_model.model_parameters["retrieval_threshold"] = -2  # chunks below this can't be retrieved
memory_model.model_parameters["instantaneous_noise"] = 0.3  # adds retrieval variability

dm = memory_model.decmem

print("Model created with subsymbolic processing enabled!")
print(f"  Decay rate: {memory_model.model_parameters['decay']}")
print(f"  Retrieval threshold: {memory_model.model_parameters['retrieval_threshold']}")
print(f"  Instantaneous noise: {memory_model.model_parameters['instantaneous_noise']}")

In [ ]:
# Add chunks at different time points to simulate learning at different ages

dm.add(paris_fact, time=0)  # just learned
dm.add(rome_fact, time=0)

dm.add(actr.makechunk(typename="fact", 
                     subject="Washington", 
                     relation="capital-of", 
                     object="USA"), 
       time=-10)

dm.add(actr.makechunk(typename="fact",
                     subject="Athens",
                     relation="capital-of",
                     object="Greece"),
       time=-100)  # learned long ago

print("Added chunks with different ages:")
print("  Paris & Rome: time=0")
print("  Washington: time=-10")
print("  Athens: time=-100")
print("\nOlder memories will have lower activation and be harder to retrieve.")

## Activation and Retrieval <a id='activation-retrieval'></a>

The activation equation in ACT-R:

**A = B + ε**

- **A** = Total activation
- **B** = Base-level activation (based on history of use)
- **ε** = Random noise

Base-level activation:

**B = ln(Σ(t_i^-d))**

- **t_i** = Time since the i-th use
- **d** = Decay parameter

In [ ]:
memory_model.productionstring(name="retrieve_capital", string="""
    =g>
        isa      goal
        state    retrieve
        country  =country
    ==>
    =g>
        state    retrieving
    +retrieval>
        isa      fact
        subject  =country
        relation capital-of
""")

memory_model.productionstring(name="report_retrieval", string="""
    =g>
        isa      goal
        state    retrieving
        country  =country
    =retrieval>
        isa      fact
        subject  =subject
        object   =capital
    ==>
    =g>
        state    done
        result   =capital
""")

memory_model.productionstring(name="retrieval_failed", string="""
    =g>
        isa      goal
        state    retrieving
        country  =country
    ?retrieval>
        state    error
    ==>
    =g>
        state    failed
""")

actr.chunktype("goal", ("state", "country", "result"))

In [ ]:
# Test retrieval

countries_to_test = ["France", "Italy", "USA", "Greece"]
retrieval_results = []

for country in countries_to_test:
    memory_model.goal.clear()
    
    memory_model.goal.add(actr.makechunk(typename="goal",
                                        state="retrieve",
                                        country=country))
    
    sim = memory_model.simulation(gui=False, trace=False)
    sim.run(2)
    
    try:
        goal_chunks = list(memory_model.goal)
        if goal_chunks:
            goal_chunk = goal_chunks[0]
            if hasattr(goal_chunk, 'result') and goal_chunk.result:
                result = goal_chunk.result
                status = "retrieved"
            elif hasattr(goal_chunk, 'state') and goal_chunk.state == "failed":
                result = None
                status = "failed"
            else:
                result = None
                status = "in progress"
        else:
            result = None
            status = "no goal"
    except:
        result = None
        status = "error"
    
    retrieval_results.append({
        'country': country,
        'result': result,
        'status': status
    })
    
    print(f"Trying to retrieve capital of {country}: {status}")
    if result:
        print(f"  Retrieved: {result}")

# Test with cities (should fail since we stored facts by country, not city)
print("\nTesting with cities (should fail):")
cities_to_test = ["Paris", "Rome"]
for city in cities_to_test:
    memory_model.goal.clear()
    memory_model.goal.add(actr.makechunk(typename="goal",
                                        state="retrieve",
                                        country=city))
    sim = memory_model.simulation(gui=False, trace=False)
    sim.run(2)
    
    goal_chunks = list(memory_model.goal)
    if goal_chunks and hasattr(goal_chunks[0], 'state'):
        status = goal_chunks[0].state
    else:
        status = "unknown"
    
    print(f"Trying to retrieve capital of {city}: {status}")

### Examining Activation Levels

Here's a more controlled example showing how activation changes with different presentation histories.

In [ ]:
activation_model = actr.ACTRModel()
activation_model.model_parameters["subsymbolic"] = True
activation_model.model_parameters["decay"] = 0.5
activation_model.model_parameters["instantaneous_noise"] = 0  # no noise for cleaner demonstration

actr.chunktype("simple_fact", "name")

fact_a = actr.makechunk(typename="simple_fact", name="A")
fact_b = actr.makechunk(typename="simple_fact", name="B")
fact_c = actr.makechunk(typename="simple_fact", name="C")

dm = activation_model.decmem

# Fact A: presented many times (strong memory)
presentation_times_a = [0, -1, -2, -5, -10, -20]
for t in presentation_times_a:
    dm.add(fact_a, time=t)

# Fact B: presented a few times (moderate memory)
presentation_times_b = [0, -5, -20]
for t in presentation_times_b:
    dm.add(fact_b, time=t)

# Fact C: presented once long ago (weak memory)  
dm.add(fact_c, time=-50)

print("Created three facts with different presentation histories:")
print(f"  Fact A: {len(presentation_times_a)} presentations")
print(f"  Fact B: {len(presentation_times_b)} presentations")
print(f"  Fact C: 1 presentation, 50 time units ago")

## Forgetting and Memory Decay <a id='forgetting'></a>

ACT-R models several well-known memory phenomena:

1. Power Law of Forgetting: Memory strength decays as a power function of time
2. Spacing Effect: Distributed practice beats massed practice
3. Retrieval Practice: Using a memory strengthens it

In [ ]:
def calculate_base_activation(times, decay=0.5, current_time=0):
    """Calculate base-level activation: B = ln(sum(t_i^-d))"""
    if not times:
        return float('-inf')
    
    sum_component = 0
    for t in times:
        time_diff = current_time - t
        if time_diff > 0:
            sum_component += time_diff ** (-decay)
    
    if sum_component > 0:
        return np.log(sum_component)
    else:
        return float('-inf')

# Simulate forgetting curves
time_points = np.arange(0, 100, 1)
activations_single = []
activations_spaced = []
activations_massed = []

for t in time_points:
    single = calculate_base_activation([0], current_time=t)
    activations_single.append(single)
    
    spaced = calculate_base_activation([0, -10, -20], current_time=t)
    activations_spaced.append(spaced)
    
    massed = calculate_base_activation([0, -1, -2], current_time=t)
    activations_massed.append(massed)

plt.figure(figsize=(10, 6))
plt.plot(time_points, activations_single, label='Single presentation', linewidth=2)
plt.plot(time_points, activations_spaced, label='Spaced practice', linewidth=2)
plt.plot(time_points, activations_massed, label='Massed practice', linewidth=2)
plt.xlabel('Time since last presentation')
plt.ylabel('Base-level activation')
plt.title('Forgetting Curves in ACT-R')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Notice:")
print("- All memories decay over time (power law of forgetting)")
print("- Multiple presentations create stronger memories")
print("- Spaced practice maintains higher activation over time")

## Real-World Application: List Learning <a id='list-learning'></a>

Here's a classic psychology experiment: learning and recalling a list of words. This demonstrates serial position effects (primacy and recency), interference between similar items, and individual differences in memory capacity.

In [ ]:
list_model = actr.ACTRModel()
list_model.model_parameters["subsymbolic"] = True
list_model.model_parameters["decay"] = 0.5
list_model.model_parameters["instantaneous_noise"] = 0.3
list_model.model_parameters["retrieval_threshold"] = -1

actr.chunktype("list_item", ("word", "position", "list_id"))

word_list = ["apple", "chair", "ocean", "music", "flower", 
             "pencil", "cloud", "guitar", "window", "coffee"]

dm = list_model.decmem
study_time = 0
presentation_duration = 1.0

print("Studying word list:")
for position, word in enumerate(word_list):
    chunk = actr.makechunk(typename="list_item",
                          word=word,
                          position=position+1,
                          list_id="list1")
    
    dm.add(chunk, time=study_time)
    print(f"  {position+1}. {word} (t={study_time})")
    
    study_time += presentation_duration

print(f"\nFinished studying {len(word_list)} words in {study_time} seconds")

In [ ]:
list_model.productionstring(name="start_recall", string="""
    =g>
        isa     goal
        state   start
        list    =list
    ==>
    =g>
        state   recalling
    +retrieval>
        isa     list_item
        list_id =list
""")

list_model.productionstring(name="recall_word", string="""
    =g>
        isa     goal
        state   recalling
        list    =list
    =retrieval>
        isa     list_item
        word    =word
        position =pos
    ==>
    =g>
        state   recalling
        recalled =word
    +retrieval>
        isa     list_item
        list_id =list
        word    ~=word  
""")

list_model.productionstring(name="recall_failed", string="""
    =g>
        isa     goal
        state   recalling
    ?retrieval>
        state   error
    ==>
    =g>
        state   done
""")

actr.chunktype("goal", ("state", "list", "recalled"))

In [ ]:
recall_delay = 5.0

list_model.goal.add(actr.makechunk(typename="goal", 
                                  state="start",
                                  list="list1"))

recalled_words = []
recalled_positions = []

print(f"\nStarting free recall after {recall_delay} second delay...\n")

import io
import sys

old_stdout = sys.stdout
sys.stdout = buffer = io.StringIO()

sim = list_model.simulation(gui=False, trace=True, initial_time=study_time + recall_delay)
sim.run(20)

trace_output = buffer.getvalue()
sys.stdout = old_stdout

# Parse trace to find recalled words
lines = trace_output.split('\n')
for i, line in enumerate(lines):
    if 'RULE FIRED: recall_word' in line:
        for j in range(max(0, i-10), min(len(lines), i+5)):
            if 'RETRIEVED:' in lines[j] and 'list_item' in lines[j]:
                chunk_str = lines[j].split('RETRIEVED: ')[1]
                if 'word=' in chunk_str and 'position=' in chunk_str:
                    word_start = chunk_str.find('word=') + 5
                    word_end = chunk_str.find(',', word_start)
                    if word_end == -1:
                        word_end = chunk_str.find(')', word_start)
                    word = chunk_str[word_start:word_end].strip()
                    
                    pos_start = chunk_str.find('position=') + 9
                    pos_end = chunk_str.find(',', pos_start)
                    if pos_end == -1:
                        pos_end = chunk_str.find(')', pos_start)
                    position = int(chunk_str[pos_start:pos_end].strip())
                    
                    if word not in recalled_words:
                        recalled_words.append(word)
                        recalled_positions.append(position)
                        print(f"Recalled: {word} (position {position})")
                break

if not recalled_words:
    print("\nNo words found in trace.")

position_counts = defaultdict(int)
for pos in recalled_positions:
    position_counts[pos] += 1

print(f"\nRecall summary:")
print(f"  Total words recalled: {len(recalled_words)}/{len(word_list)}")
if len(word_list) > 0:
    print(f"  Recall percentage: {len(recalled_words)/len(word_list)*100:.1f}%")

if recalled_positions:
    plt.figure(figsize=(10, 6))
    positions = list(range(1, len(word_list)+1))
    recall_probability = [position_counts.get(p, 0) for p in positions]
    
    plt.bar(positions, recall_probability, alpha=0.7)
    plt.xlabel('Serial Position')
    plt.ylabel('Recall Count')
    plt.title('Serial Position Effect in Free Recall')
    plt.xticks(positions)
    plt.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()
    
    print("\nSerial position effect:")
    print("- Primacy: Better recall of early items (more rehearsal)")
    print("- Recency: Better recall of recent items (still active)")
else:
    print("\nSerial position curve cannot be plotted without recall data.")

## Exercises <a id='exercises'></a>

### Exercise 1: Semantic Network

Create a semantic network of animal facts and test how activation spreads between related concepts.

In [ ]:
# Your code here
# Create chunks for animals with properties:
# - category (mammal, bird, reptile)
# - size (small, medium, large)  
# - habitat (land, water, air)

# actr.chunktype("animal", ("name", "category", "size", "habitat"))
# dog = actr.makechunk(typename="animal", name="dog", category="mammal", size="medium", habitat="land")

# Test retrieval with partial matching enabled

### Exercise 2: Interference Effects

Model proactive interference by having the model learn two similar lists and observe how the first list interferes with recalling the second.

In [ ]:
# Your code here
# Create two word lists with overlapping categories
# List 1: apple, banana, orange, grape, ...
# List 2: carrot, potato, tomato, onion, ...

# Study both lists at different times
# Try to recall list 2 and see if list 1 items intrude

### Exercise 3: Memory Palace Technique

Implement the "method of loci" (memory palace) technique by associating items with locations.

In [ ]:
# Your code here
# Create chunks that link items to locations:
# actr.chunktype("loci_item", ("item", "location", "sequence"))

# Example:
# - "apple" -> "front door"
# - "keys" -> "living room"
# - "book" -> "kitchen"

# Compare recall with and without location associations

### Exercise 4: Individual Differences

Create multiple models with different memory parameters to simulate individual differences in memory ability.

In [ ]:
# Your code here
# Create three models:
# 1. "Good memory": low decay (0.3), low noise (0.1)
# 2. "Average memory": default parameters
# 3. "Poor memory": high decay (0.7), high noise (0.5)

# Have each model learn the same list
# Compare their recall performance

## Summary

This tutorial covered:

- How chunks represent different types of knowledge
- How activation determines what we remember
- Mathematical models of memory decay
- Serial position effects and interference
- Individual differences through parameter variation

Memory in ACT-R is an active system, not just storage. The timing of learning events matters. Spaced practice beats massed practice. And adding noise makes the model more realistic.

Applications: optimizing study schedules, designing training programs, modeling memory disorders, building more human-like AI systems.

Tutorial 3 will cover production rules in depth, conflict resolution, and procedural learning.

---

## Additional Resources

Research papers:
- Anderson, J. R., & Schooler, L. J. (1991). Reflections of the environment in memory.
- Pavlik Jr, P. I., & Anderson, J. R. (2005). Practice and forgetting effects on vocabulary memory.

Advanced topics:
- Spreading activation from goals and context
- Blending for reconstructive memory
- Base-level learning from environment statistics